In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install -q pyfaidx pandas numpy tqdm


In [3]:
import os
import subprocess


def run_cmd(cmd: str):
    print(cmd)
    subprocess.run(cmd, shell=True, check=True)


if not os.path.exists("hg38.fa"):
    if not os.path.exists("hg38.fa.gz"):
        run_cmd("wget -q https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz -O hg38.fa.gz")
    run_cmd("gunzip -f hg38.fa.gz")
else:
    print("Found hg38.fa, skipping download.")

if not os.path.exists("clinvar.vcf"):
    if not os.path.exists("clinvar.vcf.gz"):
        run_cmd("wget -q https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz -O clinvar.vcf.gz")
    run_cmd("gunzip -f clinvar.vcf.gz")
else:
    print("Found clinvar.vcf, skipping download.")

print("Reference files ready.")


--2026-03-10 03:17:19--  https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
Resolving hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)|128.114.119.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 983659424 (938M) [application/x-gzip]
Saving to: ‘hg38.fa.gz’

hg38.fa.gz          100%[===================>] 938.09M  25.7MB/s    in 40s     

2026-03-10 03:18:00 (23.6 MB/s) - ‘hg38.fa.gz’ saved [983659424/983659424]



In [4]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pyfaidx import Fasta

SEED = 42
rng = np.random.default_rng(SEED)

WINDOW_SIZE = 1024
MUT_POS = WINDOW_SIZE // 2 - 1  # index 511 in a 1024 window

# Increase N_HEALTHY_TARGET to 500_000+ for full research run.
N_HEALTHY_TARGET = 200_000
N_VARIANT_TARGET = 100_000
MAX_N_FRAC = 0.02

CHROMS = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]

NUC_TO_IDX = {"A": 0, "C": 1, "G": 2, "T": 3, "N": 4}
IDX_TO_NUC = {v: k for k, v in NUC_TO_IDX.items()}
COMP_MAP = np.array([3, 2, 1, 0, 4], dtype=np.int8)  # A<->T, C<->G, N->N

BYTE_LUT = np.full(256, 4, dtype=np.int8)
for base, idx in NUC_TO_IDX.items():
    BYTE_LUT[ord(base)] = idx


def seq_to_tokens(seq: str) -> np.ndarray:
    arr = np.frombuffer(seq.encode("ascii"), dtype=np.uint8)
    return BYTE_LUT[arr]


def reverse_complement_tokens(x: np.ndarray) -> np.ndarray:
    return COMP_MAP[x[:, ::-1]]


genome = Fasta("hg38.fa", as_raw=True, sequence_always_upper=True)
valid_chroms = [c for c in CHROMS if c in genome.keys()]
chrom_lengths = {c: len(genome[c]) for c in valid_chroms}
chrom_capacity = np.array([max(0, chrom_lengths[c] - WINDOW_SIZE) for c in valid_chroms], dtype=np.float64)
chrom_probs = chrom_capacity / chrom_capacity.sum()

print(f"Genome loaded: {len(valid_chroms)} chromosomes")
print(f"Window size: {WINDOW_SIZE}, mutation index: {MUT_POS}")


--2026-03-10 03:20:24--  https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.11, 130.14.250.10, 2607:f220:41e:250::11, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.11|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 190285755 (181M) [application/x-gzip]
Saving to: ‘clinvar.vcf.gz’

clinvar.vcf.gz      100%[===================>] 181.47M  14.0MB/s    in 15s     

2026-03-10 03:20:39 (12.3 MB/s) - ‘clinvar.vcf.gz’ saved [190285755/190285755]



In [5]:
def sample_unbiased_healthy_windows(
    target_n: int,
    window_size: int = WINDOW_SIZE,
    max_n_frac: float = MAX_N_FRAC,
) -> np.ndarray:
    """Sample random genome windows weighted by chromosome length."""
    x = np.empty((target_n, window_size), dtype=np.int8)

    kept = 0
    attempts = 0
    max_attempts = target_n * 50

    pbar = tqdm(total=target_n, desc="Sampling healthy hg38 windows")
    while kept < target_n and attempts < max_attempts:
        chrom_idx = int(rng.choice(len(valid_chroms), p=chrom_probs))
        chrom = valid_chroms[chrom_idx]
        chrom_len = chrom_lengths[chrom]

        if chrom_len <= window_size:
            attempts += 1
            continue

        start0 = int(rng.integers(0, chrom_len - window_size))
        end0 = start0 + window_size
        seq = genome[chrom][start0:end0]
        tokens = seq_to_tokens(seq)

        if tokens.shape[0] != window_size:
            attempts += 1
            continue

        if (tokens == 4).mean() > max_n_frac:
            attempts += 1
            continue

        x[kept] = tokens
        kept += 1
        attempts += 1
        pbar.update(1)

    pbar.close()

    if kept < target_n:
        print(f"Warning: kept {kept}/{target_n}. Increase attempts or relax MAX_N_FRAC.")
        x = x[:kept]

    return x


X_healthy_tokens = sample_unbiased_healthy_windows(N_HEALTHY_TARGET)
print("Healthy tokens shape:", X_healthy_tokens.shape)
print("Healthy N fraction :", float((X_healthy_tokens == 4).mean()))


Parsing VCF: 4397738it [00:09, 475504.16it/s]


Extracted 351596 usable variants!


In [6]:
def parse_clinvar_snps(vcf_path: str) -> pd.DataFrame:
    rows = []
    allowed_chroms = [str(i) for i in range(1, 23)] + ["X", "Y"]

    with open(vcf_path, "r") as f:
        for line in tqdm(f, desc="Parsing ClinVar VCF"):
            if line.startswith("#"):
                continue

            parts = line.rstrip("\n").split("\t")
            chrom = parts[0]
            pos = int(parts[1])
            ref = parts[3]
            alt = parts[4]
            info = parts[7]

            if chrom not in allowed_chroms:
                continue
            if len(ref) != 1:
                continue
            if "," in alt or len(alt) != 1:
                continue
            if ref not in NUC_TO_IDX or alt not in NUC_TO_IDX:
                continue

            if "CLNSIG=Pathogenic" in info or "CLNSIG=Likely_pathogenic" in info:
                label = 1
            elif "CLNSIG=Benign" in info or "CLNSIG=Likely_benign" in info:
                label = 0
            else:
                continue

            rows.append((f"chr{chrom}", pos, ref, alt, label))

    return pd.DataFrame(rows, columns=["chrom", "pos", "ref", "alt", "label"])


variants_df = parse_clinvar_snps("clinvar.vcf")
print("Usable ClinVar SNPs:", len(variants_df))

if N_VARIANT_TARGET is not None and len(variants_df) > N_VARIANT_TARGET:
    variants_df = variants_df.sample(N_VARIANT_TARGET, random_state=SEED).reset_index(drop=True)


def build_eval_pairs(df: pd.DataFrame):
    ref_windows = []
    alt_windows = []
    labels = []
    variant_position = []
    ref_base_idx = []
    alt_base_idx = []
    variant_chrom = []
    variant_pos = []

    for row in tqdm(df.itertuples(index=False), total=len(df), desc="Building eval windows"):
        chrom = row.chrom
        if chrom not in genome.keys():
            continue

        pos1 = int(row.pos)
        start0 = pos1 - 1 - MUT_POS
        end0 = start0 + WINDOW_SIZE

        if start0 < 0:
            continue

        seq = genome[chrom][start0:end0]
        if len(seq) != WINDOW_SIZE:
            continue

        ref_tokens = seq_to_tokens(seq)
        r_idx = NUC_TO_IDX[row.ref]
        a_idx = NUC_TO_IDX[row.alt]

        if ref_tokens[MUT_POS] != r_idx:
            continue

        if (ref_tokens == 4).mean() > MAX_N_FRAC:
            continue

        alt_tokens = ref_tokens.copy()
        alt_tokens[MUT_POS] = a_idx

        ref_windows.append(ref_tokens)
        alt_windows.append(alt_tokens)
        labels.append(int(row.label))
        variant_position.append(MUT_POS)
        ref_base_idx.append(r_idx)
        alt_base_idx.append(a_idx)
        variant_chrom.append(chrom)
        variant_pos.append(pos1)

    return (
        np.asarray(ref_windows, dtype=np.int8),
        np.asarray(alt_windows, dtype=np.int8),
        np.asarray(labels, dtype=np.int8),
        np.asarray(variant_position, dtype=np.int16),
        np.asarray(ref_base_idx, dtype=np.int8),
        np.asarray(alt_base_idx, dtype=np.int8),
        np.asarray(variant_chrom),
        np.asarray(variant_pos, dtype=np.int32),
    )


(
    X_eval_ref_tokens,
    X_eval_alt_tokens,
    Y_labels,
    variant_position,
    ref_base_idx,
    alt_base_idx,
    variant_chrom,
    variant_pos,
) = build_eval_pairs(variants_df)

print("Eval ref shape:", X_eval_ref_tokens.shape)
print("Eval alt shape:", X_eval_alt_tokens.shape)
print("Pathogenic fraction:", float(Y_labels.mean()))


100%|██████████| 50000/50000 [00:03<00:00, 13448.24it/s]

Successfully generated 12263 window pairs.


In [7]:
# Reverse-complement augmentation for healthy training windows
X_healthy_rc_tokens = reverse_complement_tokens(X_healthy_tokens)
X_healthy_aug_tokens = np.concatenate([X_healthy_tokens, X_healthy_rc_tokens], axis=0)

print("Healthy (original) shape:", X_healthy_tokens.shape)
print("Healthy (augmented) shape:", X_healthy_aug_tokens.shape)


def print_token_distribution(x: np.ndarray, name: str):
    counts = np.bincount(x.reshape(-1).astype(np.int64), minlength=5)
    freqs = counts / counts.sum()
    print(f"{name} token freq [A,C,G,T,N]:", np.round(freqs, 4).tolist())


print_token_distribution(X_healthy_aug_tokens, "Healthy augmented")
print_token_distribution(X_eval_ref_tokens, "Eval reference")


Tokenizing: 100%|██████████| 12263/12263 [00:00<00:00, 21774.81it/s]


Healthy shape: (12263, 1024)
Corrupted shape: (12263, 1024)


In [8]:
import os
from numpy.lib.format import open_memmap

OUT_DIR = '/kaggle/working/processed_data'
os.makedirs(OUT_DIR, exist_ok=True)


def save_onehot_memmap(tokens: np.ndarray, out_path: str, chunk_size: int = 20000):
    """Save one-hot array [N, L, 4] as uint8 without loading full output in RAM."""
    n, l = tokens.shape
    out = open_memmap(out_path, mode='w+', dtype=np.uint8, shape=(n, l, 4))
    eye4 = np.eye(4, dtype=np.uint8)

    for start in tqdm(range(0, n, chunk_size), desc=f"One-hot {os.path.basename(out_path)}"):
        end = min(start + chunk_size, n)
        chunk = tokens[start:end]

        valid = (chunk >= 0) & (chunk < 4)
        clipped = np.clip(chunk, 0, 3)
        encoded = eye4[clipped]  # [B, L, 4]
        encoded[~valid] = 0

        out[start:end] = encoded

    del out


# Backward-compatible names expected by older training notebooks
np.save(f"{OUT_DIR}/X_healthy.npy", X_healthy_aug_tokens)
np.save(f"{OUT_DIR}/X_corrupted.npy", X_eval_alt_tokens)
np.save(f"{OUT_DIR}/Y_labels.npy", Y_labels)

# Explicit token datasets
np.save(f"{OUT_DIR}/X_healthy_tokens.npy", X_healthy_aug_tokens)
np.save(f"{OUT_DIR}/X_eval_ref_tokens.npy", X_eval_ref_tokens)
np.save(f"{OUT_DIR}/X_eval_alt_tokens.npy", X_eval_alt_tokens)

# Metadata for exact-position scoring
np.save(f"{OUT_DIR}/variant_position.npy", variant_position)
np.save(f"{OUT_DIR}/ref_base.npy", ref_base_idx)
np.save(f"{OUT_DIR}/alt_base.npy", alt_base_idx)
np.save(f"{OUT_DIR}/variant_chrom.npy", variant_chrom)
np.save(f"{OUT_DIR}/variant_pos.npy", variant_pos)

# One-hot exports (Phase 1 requirement)
save_onehot_memmap(X_healthy_aug_tokens, f"{OUT_DIR}/X_healthy_onehot.npy")
save_onehot_memmap(X_eval_ref_tokens, f"{OUT_DIR}/X_eval_ref_onehot.npy")
save_onehot_memmap(X_eval_alt_tokens, f"{OUT_DIR}/X_eval_alt_onehot.npy")

print("Phase 1 complete.")
print(f"Saved dataset to: {OUT_DIR}")
print("Use X_healthy_onehot.npy for one-hot training data.")
print("Use X_eval_ref/alt_onehot.npy + Y_labels.npy for evaluation.")


Phase 1 Complete! Data saved to /kaggle/working/processed_data/
